# Chapitre 17 - Exercices 17.7 à 17.12

Exercices de programmation sur le modèle graphique gaussien (17.7 : algorithme 17.1,
17.8 : lasso et graphical lasso, 17.9 : algorithme EM avec valeurs manquantes), puis
exercices sur le modèle de Ising (17.10 : nœud constant, 17.11 : lois conditionnelles
logistiques, 17.12 : lien avec la régression de Poisson).

**Données.** Les exercices 17.7–17.9 utilisent les données de cytométrie en flux de
Sachs et al. (11 protéines, 7466 cellules), qui ne sont pas disponibles ici. Nous les
remplaçons par des données gaussiennes **simulées** de même taille dont le vrai graphe
est celui de la figure 17.1 (`g.simulate_cytometry_like`). Les noms de protéines ne
sont que des étiquettes. Avantage : on connaît la vérité et on peut juger les
estimations.

In [1]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import ch17_graphes as g

In [2]:
Xc, Theta_vrai, A_fig = g.simulate_cytometry_like(N=7466, seed=0)
noms, _, pos = g.cytometry_like_graph()
N, p = Xc.shape
Sc = g.empirical_covariance(Xc)
print("N = %d, p = %d, arêtes du graphe de la figure 17.1 : %d" % (N, p, A_fig.sum() // 2))

N = 7466, p = 11, arêtes du graphe de la figure 17.1 : 16


## Exercice 17.7 : Programmer l'algorithme 17.1

**Énoncé.** Écrire un programme mettant en œuvre la procédure de régressions modifiées
(algorithme 17.1) pour ajuster un modèle graphique gaussien dont certaines arêtes sont
imposées absentes. L'essayer sur les données de cytométrie avec le graphe de la
figure 17.1.

**Rappels.** On maximise $\log\det\Theta - \operatorname{tr}(S\Theta)$ sous $\theta_{jk} = 0$ pour
$(j,k)\notin E$. Pour chaque variable $j$, on résout le système réduit aux voisins
$W^*_{11}\beta^* = s^*_{12}$ (éq. (17.19)), on complète $\beta$ par des zéros et on met à jour
$w_{12} = W_{11}\hat\beta$ ; à la fin, $\hat\theta_{22} = 1/(s_{22} - w_{12}^T\hat\beta)$ et
$\hat\theta_{12} = -\hat\beta\,\hat\theta_{22}$ (exercice 17.6).

In [3]:
def algorithme_17_1(S, A, tol=1e-10, max_cycles=1000):
    """Version commentée de l'algorithme 17.1 (régressions modifiées)."""
    p = S.shape[0]
    W = S.copy()                                   # étape 1
    betas = np.zeros((p, p))                       # betas[:, j] : régression de j
    for cycle in range(max_cycles):                # étape 2
        W_avant = W.copy()
        for j in range(p):
            autres = np.array([k for k in range(p) if k != j])
            vois = autres[A[j, autres]]            # variables reliées à j
            beta = np.zeros(p)
            if len(vois) > 0:                      # (b) système réduit (17.19)
                beta[vois] = np.linalg.solve(W[np.ix_(vois, vois)], S[vois, j])
            betas[:, j] = beta
            w12 = W[np.ix_(autres, autres)] @ beta[autres]   # (c)
            W[autres, j] = w12
            W[j, autres] = w12
        if np.abs(W - W_avant).max() < tol:
            break
    Theta = np.zeros((p, p))                       # étape 3
    for j in range(p):
        autres = [k for k in range(p) if k != j]
        t22 = 1.0 / (S[j, j] - W[autres, j] @ betas[autres, j])
        Theta[j, j] = t22
        Theta[autres, j] = -betas[autres, j] * t22
    return W, (Theta + Theta.T) / 2, cycle + 1


W17, T17, n_cycles = algorithme_17_1(Sc, A_fig)
W_mod, T_mod = g.fit_known_graph(Sc, A_fig)
print("convergence en %d cycles ; écart avec esl.ch17_graphes : %.1e" % (n_cycles, np.abs(T17 - T_mod).max()))

convergence en 3 cycles ; écart avec esl.ch17_graphes : 0.0e+00


**Vérifications des propriétés de la solution.**

1. $\hat\theta_{jk} = 0$ pour toute paire hors du graphe ;
2. $\hat w_{jk} = s_{jk}$ sur la diagonale et sur les arêtes (car $\Gamma$ y est nul dans
   (17.13)) : le modèle reproduit exactement les covariances observées des paires reliées ;
3. $\hat W\hat\Theta = I$.